In [1]:
import dspy

raw_data = [
    ("Die Bildqualität ist hervorragend und die Bedienung intuitiv.", "positiv"),
    ("Ich bin sehr zufrieden mit dem Produkt, es übertrifft meine Erwartungen.", "positiv"),
    ("Das Preis-Leistungs-Verhältnis ist unschlagbar.", "positiv"),
    ("Leider hat das Gerät nach kurzer Zeit den Geist aufgegeben.", "negativ"),
    ("Der Kundenservice war überhaupt nicht hilfreich und unfreundlich.", "negativ"),
    ("Die Akkulaufzeit ist enttäuschend kurz.", "negativ"),
    ("Das Produkt wurde pünktlich geliefert.", "neutral"),
    ("Die Verpackung war angemessen.", "neutral"),
    ("Die Farbe des Produkts entspricht der Abbildung online.", "neutral"),
    ("Ein wirklich tolles Erlebnis von Anfang bis Ende!", "positiv"),
    ("Ich würde dieses Produkt niemandem empfehlen.", "negativ"),
    ("Die Anleitung ist schwer zu verstehen.", "negativ"),
]

In [2]:
# Umwandlung der Rohdaten in dspy.Example-Objekte
dspy_examples = [
    dspy.Example(text=text, sentiment=sentiment).with_inputs("text")
    for text, sentiment in raw_data
]

# Aufteilung des Datensatzes in Trainings- und Evaluationsset (80/20-Split)
split_index = int(len(dspy_examples) * 0.8)
trainset = dspy_examples[:split_index]
devset = dspy_examples[split_index:]

print(f"Anzahl der Beispiele im Trainingsset: {len(trainset)}")
print(f"Anzahl der Beispiele im Evaluationsset: {len(devset)}")

Anzahl der Beispiele im Trainingsset: 9
Anzahl der Beispiele im Evaluationsset: 3


In [3]:
# Signatur für die Sentiment-Klassifizierung
class SentimentSignature(dspy.Signature):
    """Klassifiziert den Sentiment eines gegebenen Textes als positiv, negativ oder neutral."""
    text = dspy.InputField(desc="Der zu klassifizierende Text.")
    sentiment = dspy.OutputField(desc="Das Ergebnis der Klassifizierung: positiv, negativ oder neutral.")

# Modul, das die Klassifizierung durchführt
class SimpleClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.Predict(SentimentSignature)

    def forward(self, text):
        return self.predictor(text=text)

In [4]:
# Konfiguration des lokalen Sprachmodells
local_llm = dspy.LM(
    "openai/qwen3:4b", 
    api_base="http://localhost:11434/v1", 
    api_key="no_key_needed",
    temperature=2,
    cache=False
)

dspy.configure(lm=local_llm)

In [5]:
# Metrik zur Evaluation: Exakte Übereinstimmung
def validation_metric(gold, pred, trace=None):
    # 'gold' ist das Beispiel aus dem devset, 'pred' ist die Vorhersage des Modells
    return gold.sentiment.lower() == pred.sentiment.lower()

In [6]:
from dspy.teleprompt import BootstrapFewShot

# Konfiguration des Optimizers
config = dict(max_bootstrapped_demos=3, max_labeled_demos=3)

# Instanziierung des Optimizers
optimizer = BootstrapFewShot(metric=validation_metric, **config)

# Kompilierung des Modells
optimized_classifier = optimizer.compile(SimpleClassifier(), trainset=trainset)

 33%|████████████████████████████████████████████████████                                                                                                        | 3/9 [01:14<02:29, 24.89s/it]

Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.


In [7]:
from dspy.evaluate import Evaluate

# Evaluator instanziieren
evaluator = Evaluate(devset=devset, num_threads=1, display_progress=True, display_table=5)

# Evaluation des unoptimierten Klassifikators
unoptimized_classifier = SimpleClassifier()
evaluator(unoptimized_classifier, metric=validation_metric)

# Evaluation des optimierten Klassifikators
evaluator(optimized_classifier, metric=validation_metric)

Average Metric: 3.00 / 3 (100.0%): 100%|████████████████████████████████████| 3/3 [00:19<00:00,  6.41s/it]

2025/12/17 07:17:51 INFO dspy.evaluate.evaluate: Average Metric: 3 / 3 (100.0%)


,text,example_sentiment,pred_sentiment,validation_metric
0,Ein wirklich tolles Erlebnis von Anfang bis Ende!,positiv,positiv,✔️ [True]
1,Ich würde dieses Produkt niemandem empfehlen.,negativ,negativ,✔️ [True]
2,Die Anleitung ist schwer zu verstehen.,negativ,negativ,✔️ [True]


Average Metric: 3.00 / 3 (100.0%): 100%|████████████████████████████████████| 3/3 [00:27<00:00,  9.03s/it]

2025/12/17 07:18:19 INFO dspy.evaluate.evaluate: Average Metric: 3 / 3 (100.0%)


,text,example_sentiment,pred_sentiment,validation_metric
0,Ein wirklich tolles Erlebnis von Anfang bis Ende!,positiv,positiv,✔️ [True]
1,Ich würde dieses Produkt niemandem empfehlen.,negativ,negativ,✔️ [True]
2,Die Anleitung ist schwer zu verstehen.,negativ,negativ,✔️ [True]


EvaluationResult(score=100.0, results=<list of 3 results>)

In [8]:
# Testvorhersage mit dem optimierten Modell, um den Prompt zu inspizieren
test_text = "Der Versand hat etwas länger gedauert, aber das Produkt ist gut."
optimized_classifier(text=test_text)

# Anzeige des letzten generierten Prompts
local_llm.inspect_history(n=1)





[2025-12-17T07:18:30.878614]

System message:

Your input fields are:
1. `text` (str): Der zu klassifizierende Text.
Your output fields are:
1. `sentiment` (str): Das Ergebnis der Klassifizierung: positiv, negativ oder neutral.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Klassifiziert den Sentiment eines gegebenen Textes als positiv, negativ oder neutral.


User message:

[[ ## text ## ]]
Die Bildqualität ist hervorragend und die Bedienung intuitiv.


Assistant message:

[[ ## sentiment ## ]]
positiv

[[ ## completed ## ]]


User message:

[[ ## text ## ]]
Ich bin sehr zufrieden mit dem Produkt, es übertrifft meine Erwartungen.


Assistant message:

[[ ## sentiment ## ]]
positiv

[[ ## completed ## ]]


User message:

[[ ## text ## ]]
Das Preis-Leistungs-Verhältnis ist unschlagbar.


Assist

In [9]:
# Testvorhersage mit dem nicht optimierten Modell, um den Prompt zu inspizieren
test_text = "Der Versand hat etwas länger gedauert, aber das Produkt ist gut."
unoptimized_classifier(text=test_text)

# Anzeige des letzten generierten Prompts
local_llm.inspect_history(n=1)





[2025-12-17T07:19:06.076388]

System message:

Your input fields are:
1. `text` (str): Der zu klassifizierende Text.
Your output fields are:
1. `sentiment` (str): Das Ergebnis der Klassifizierung: positiv, negativ oder neutral.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Klassifiziert den Sentiment eines gegebenen Textes als positiv, negativ oder neutral.


User message:

[[ ## text ## ]]
Der Versand hat etwas länger gedauert, aber das Produkt ist gut.

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
positiv

[[ ## completed ## ]]
Es ist positiv geschnitten, weil das Produkt direkt „gut" beschrieben wird (streng positive Ausführung), während das läng